In [0]:
%sql
-- Garantindo que o schema silver esteja inserido no catálogo medalhao
USE CATALOG medalhao;

-- Criando o schema silver
CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
#Importando a biblioteca Window do pyspark para utilizar o Window Functions para ter apenas o registro mais recente mantido 
from pyspark.sql import Window
from pyspark.sql import functions as F

#Trazendo os dados brutos da camada bronze
df_customers = spark.table("medalhao.bronze.tb_customers")

#criando o filtro (a limpeza) que será responsável por remover as duplicações do mesmo cliente
#partitionBy criará pastas independentes de cada cliente com todos seus acessos
#orderBy irá ordenar os dados por data de acesso e o de interesse é o mais recente 
windows_spec = Window.partitionBy("customer_id").orderBy(F.col("timestamp_ingestion").desc())

#Realizando o mapeamento do dataframe "customer"
#O withColumn será responsável por criar uma coluna de contagem que irá organizar os clientes sem repetições com acessos mais recentes usando o filtro criado anteriormente

df_silver_costumer = (
    df_customers
    .withColumn("row_num", F.row_number().over(windows_spec))
    .filter("row_num == 1")
    .select(
        F.col("customer_id").alias("id_consumidor"),
        F.col("customer_unique_id").alias("id_unico_do_cliente"),
        F.col("customer_zip_code_prefix").alias("prefixo_cep"),
        F.upper(F.col("customer_city")).alias("cidade"),
        F.upper(F.col("customer_state")).alias("estado"),
        F.col("customer_name").alias("nome"),
        F.col("customer_gender").alias("genero"),
        F.col("customer_birth_date").alias("data_de_nascimento"),
        F.col("customer_age").alias("idade")
    )
)
#Usando o overwriteSchema para que o Job não quebre caso a estrutura da Bronze mude
df_silver_costumer.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medalhao.silver.dim_consumidores")


In [0]:
# Importando as funções e a biblioteca Window do PySpark para realizar a limpeza e deduplicação
from pyspark.sql import functions as F
from pyspark.sql import Window

# Buscando os dados brutos de pedidos na camada Bronze
df_bronze_orders = spark.table("medalhao.bronze.tb_orders")

# Fiz essa limpeza de duplicatas, pois vi nos testes que alguns pedidos vinham repetidos e isso ia acabar sujando os números finais de vendas.
# Criando a regra de limpeza: partitionBy agrupa por cada pedido único (order_id)
# O orderBy garante que, se houver duplicatas, a carga mais recente será mantida (timestamp_ingestion)
windows_spec = Window.partitionBy("order_id").orderBy(F.col("timestamp_ingestion").desc())

# Iniciando o mapeamento e as transformações da camada Silver
df_silver_orders = (
    df_bronze_orders
    # Criação de um índice numérico para identificar o registro mais atual de cada pedido
    .withColumn("row_num", F.row_number().over(windows_spec))
    # Mantendo apenas o primeiro registro (o mais recente) e descartando duplicatas
    .filter("row_num == 1")
    # Seleção e renomeação de colunas (aliasing) para o padrão de negócio
    # Uso do to_timestamp para garantir que as datas sejam tratadas como tempo e não texto
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("customer_id").alias("id_consumidor"),
        F.col("order_status").alias("status_pedido"),
        F.to_timestamp("order_purchase_timestamp").alias("data_da_compra_do_pedido"),
        F.to_timestamp("order_approved_at").alias("pedido_aprovado_em"),
        F.to_timestamp("order_delivered_carrier_date").alias("data_de_entrega_pela_transportadora"),
        F.to_timestamp("order_delivered_customer_date").alias("data_de_entrega_ao_cliente"),
        F.to_timestamp("order_estimated_delivery_date").alias("data_estimada_de_entrega")
    )
    # Calculando a métrica de Lead Time: diferença real em dias entre a compra e a entrega
    .withColumn("tempo_entrega_dias", F.datediff(F.col("data_de_entrega_ao_cliente"), F.col("data_da_compra_do_pedido")))
    # Calculando o tempo que a Olist prometeu ao cliente 
    .withColumn("tempo_entrega_estimado_dias", F.datediff(F.col("data_estimada_de_entrega"), F.col("data_da_compra_do_pedido")))
    # Calculando o atraso ou antecipação (diferença entre o real e o estimado)
    .withColumn("diferenca_entrega_dias", F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias"))
    # Criando uma flag de status de logística para análise de performance da entrega
    .withColumn("entrega_no_prazo", 
                F.when(F.col("status_pedido") != "delivered", "Não Entregue")
                 .when(F.col("data_de_entrega_ao_cliente") <= F.col("data_estimada_de_entrega"), "Sim")
                 .otherwise("Não"))
)
#Usando o overwriteSchema para que o Job não quebre caso a estrutura da Bronze mude
df_silver_orders.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medalhao.silver.fat_pedidos")

In [0]:
# Importando as bibliotecas de funções e janelas para processamento de dados
from pyspark.sql import functions as F
from pyspark.sql import Window

# Carregando os dados brutos de itens dos pedidos da camada Bronze
df_bronze_orders_items = spark.table("medalhao.bronze.tb_order_items")

# Fiz essa limpeza de duplicatas, pois se deixasse passar itens repetidos no mesmo pedido, o valor total da compra ia  para cima sem necessidade e no fim ia dar erro na conta do financeiro.
# partitionBy agrupa pela combinação de Pedido e Item para garantir a unicidade da linha
# orderBy utiliza o timestamp de ingestão para selecionar sempre a carga mais recente
windows_spec = Window.partitionBy("order_id", "order_item_id").orderBy(F.col("timestamp_ingestion").desc())

# Iniciando o mapeamento e as transformações para a camada Silver
df_silver_items = (
    df_bronze_orders_items
    # Criando a coluna de numeração para identificar o registro mais atualizado
    .withColumn("row_num", F.row_number().over(windows_spec))
    # Mantendo apenas o registro mais novo 
    .filter("row_num == 1")
    # Selecionando e renomeando as colunas conforme o padrão de negócio
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("order_item_id").alias("id_item"),
        F.col("product_id").alias("id_produto"),
        F.col("seller_id").alias("id_vendedor"),
        F.col("shipping_limit_date").alias("data_limite_de_envio"),
        # Arredondando os valores monetários para 2 casas decimais
        F.round(F.col("price"), 2).alias("valor"),
        F.round(F.col("freight_value"), 2).alias("valor_frete")
    )
    # Calculando o valor total do item (Preço + Frete) para facilitar análises do valor final total
    .withColumn("valor_total_item", F.round(F.col("valor") + F.col("valor_frete"), 2))
)
#Usando o overwriteSchema para que o Job não quebre caso a estrutura da Bronze mude
df_silver_items.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medalhao.silver.fat_itens_pedidos")

In [0]:
# Importando as ferramentas de funções e janelas para o tratamento dos dados
from pyspark.sql import functions as F
from pyspark.sql import Window

# Buscando a tabela bruta de pagamentos na camada Bronze
df_bronze_order_payments = spark.table("medalhao.bronze.tb_order_payments")

# Fiz essa limpeza de duplicatas, pois percebi que se não fizesse isso, o Spark ia contar a mesma parcela de cartão duas vezes e ia parecer que o cliente pagou diferente do valor real.
# Agrupamos por id_pedido e sequencial do pagamento para não perder nenhuma parcela
# Ordenamos pela data de ingestão mais recente para manter o dado mais atualizado
windows_spec = Window.partitionBy("order_id", "payment_sequential").orderBy(F.col("timestamp_ingestion").desc())

# Iniciando o mapeamento e a tradução dos dados para a camada Silver
df_silver_payments = (
    df_bronze_order_payments
    # Criando a coluna de contagem para filtrar duplicatas através da Window Function
    .withColumn("row_num", F.row_number().over(windows_spec))
    # Mantendo apenas o primeiro registro (o mais atual) e removendo repetições
    .filter(F.col("row_num") == 1)
    # Selecionando e renomeando as colunas para o padrão de negócio
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("payment_sequential").alias("pagamento_sequencial"),
        # Aplicando a tradução dos tipos de pagamento de inglês para português
        # O uso do 'F.when' permite criar uma lógica condicional para cada categoria
        F.when(F.col("payment_type") == "credit_card", "cartao_credito")
         .when(F.col("payment_type") == "boleto", "boleto")
         .when(F.col("payment_type") == "voucher", "vale_presente")
         .when(F.col("payment_type") == "debit_card", "cartao_debito")
         .when(F.col("payment_type") == "not_defined", "nao_definido")
         .otherwise("outro").alias("tipo_pagamento"),
        F.col("payment_installments").alias("pagamento_parcelas"),
        # Arredondando os valores para exatamente 2 casas decimais (padrão financeiro)
        F.round(F.col("payment_value"), 2).alias("valor_pagamento")
    )
)
#Usando o overwriteSchema para que o Job não quebre caso a estrutura da Bronze mude
df_silver_payments.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medalhao.silver.fat_pagamentos_pedido")

In [0]:
# Importando as ferramentas de funções e janelas para o tratamento dos dados
from pyspark.sql import functions as F
from pyspark.sql import Window

# Carregando os dados brutos de avaliações da camada Bronze
df_bronze_reviews = spark.table("medalhao.bronze.tb_order_reviews")

# Configurando a regra de limpeza para garantir a unicidade da avaliação por pedido
# Ordenando pela ingestão mais recente para manter sempre o comentário mais atualizado
windows_spec = Window.partitionBy("review_id", "order_id").orderBy(F.col("timestamp_ingestion").desc())

# Iniciando o processo de tratamento e mapeamento para a camada Silver
df_silver_reviews = (
    df_bronze_reviews
    # Criando a coluna de numeração para remover duplicatas através da Window Function
    .withColumn("row_num", F.row_number().over(windows_spec))
    .filter(F.col("row_num") == 1)
    
    # Acabei por realizar uma etapa de "blindagem" depois que após testes alguns casos os comentários acabavam por invadir seções que não pertenciam a ele como de id's e titulo
    # Filtrei apenas IDs com exatos 32 caracteres e sem espaços. Isso fez eliminar linhas onde o comentário "invadiu" a coluna do ID devido a erros de delimitador no CSV
    .filter(F.length(F.col("order_id")) == 32) 
    .filter(~F.col("order_id").contains(" ")) 
    
    # Seleção e tratamento de colunas com limpeza de caracteres especiais
    .select(
        F.col("review_id").alias("id_avaliacao"),
        F.col("order_id").alias("id_pedido"),

        # Convertendo a nota para Inteiro para permitir cálculos matemáticos na camada de análise
        F.col("review_score").cast("int").alias("nota_avaliacao"),
        
        # regexp_replace: Substitui quebras de linha (\n ou \r) por espaços para não quebrar a estrutura da tabela
        # coalesce: Caso o título ou comentário venha nulo, preenchemos com um texto padrão amigável
        F.coalesce(
            F.regexp_replace(F.col("review_comment_title"), r"[\n\r]", " "), 
            F.lit("Sem título")
        ).alias("titulo_avaliacao"),
        
        F.coalesce(
            F.regexp_replace(F.col("review_comment_message"), r"[\n\r]", " "), 
            F.lit("Sem comentário")
        ).alias("comentario_avaliacao"),

        # try_to_timestamp: Tenta converter as datas, retornando nulo em caso de falha
        F.try_to_timestamp(F.col("review_creation_date")).alias("data_criacao_avaliacao"),
        F.try_to_timestamp(F.col("review_answer_timestamp")).alias("data_resposta_avaliacao")
    )

    # Garantindo que apenas registros com notas e datas válidas sigam para a Silver
    .filter(F.col("nota_avaliacao").isNotNull())
    .filter(F.col("data_criacao_avaliacao").isNotNull())
    # Impede que avaliações com datas futuras entrem na tabela final
    .filter(F.col("data_criacao_avaliacao") <= F.current_timestamp())
)
#Usando o overwriteSchema para que o Job não quebre caso a estrutura da Bronze mude
df_silver_reviews.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medalhao.silver.fat_avaliacoes_pedidos")


In [0]:
# Importando as funções e a biblioteca Window para organizar a limpeza dos dados
from pyspark.sql import functions as F
from pyspark.sql import Window

# Carregando os dados brutos de produtos da camada Bronze
df_bronze_products = spark.table("medalhao.bronze.tb_products")

# Fiz essa limpeza de IDs, pois para garantir que o catálogo fique correto. Assim, cada produto tem só uma versão com o peso e as fotos certas, sem mistura de dados antigos.
# Criando a regra de limpeza: partitionBy garante que cada produto (product_id) seja único
# O orderBy utiliza o timestamp de ingestão para selecionar a versão mais recente cadastrada
windows_spec = Window.partitionBy("product_id").orderBy(F.col("timestamp_ingestion").desc())

# Iniciando o mapeamento e a estruturação da dimensão de produtos na camada Silver
df_silver_products = (
    df_bronze_products
    # Criando um índice de linha para identificar e filtrar as duplicatas de produtos
    .withColumn("row_num", F.row_number().over(windows_spec))
    # Mantendo apenas o registro mais novo (limpeza do catálogo)
    .filter(F.col("row_num") == 1)
    # Selecionando e renomeando as colunas para o padrão descritivo da camada Silver
    .select(
        F.col("product_id").alias("id_produto"),
        F.col("product_name").alias("nome_produto"),
        F.col("product_category_name").alias("categoria_produto"),
        F.col("product_name_lenght").alias("tamanho_nome_produto"),
        F.col("product_description_lenght").alias("tamanho_descricao_produto"),
        F.col("product_photos_qty").alias("quantidade_fotos"),
        # Mapeamento de atributos físicos essenciais para cálculos de frete e logística
        F.col("product_weight_g").alias("peso_produto_gramas"),
        F.col("product_length_cm").alias("comitente_comprimento_centimetros"),
        F.col("product_height_cm").alias("altura_centimetros"),
        F.col("product_width_cm").alias("largura_centimetros")
    )
     #tratamento feito para casos de categorias que retornaram null, mas para isso não acontecer coloquei para comentar como não informado
    .fillna({"categoria_produto": "Nao Informado"})
)
#Usando o overwriteSchema para que o Job não quebre caso a estrutura da Bronze mude
df_silver_products.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medalhao.silver.dim_produtos")

In [0]:
# Importando as bibliotecas de funções e janelas para padronização e limpeza
from pyspark.sql import functions as F
from pyspark.sql import Window

# Carregando os dados brutos de vendedores da camada Bronze
df_bronze_sellers = spark.table("medalhao.bronze.tb_sellers")

# Fiz essa limpeza para garantir que cada vendedor seja único fiz para não ter risco de um vendedor aparecer com dois endereços diferentes e bugar os filtros do dashboard.
# Criando a regra de limpeza: partitionBy agrupa por cada vendedor (seller_id)
# O orderBy garante que manteremos apenas o registro com a carga mais recente (timestamp_ingestion)
windows_spec = Window.partitionBy("seller_id").orderBy(F.col("timestamp_ingestion").desc())

# Iniciando o mapeamento e a normalização de dados para a camada Silver
df_silver_sellers = (
    df_bronze_sellers
    # Criando um índice numérico para selecionar a versão mais atualizada de cada vendedor
    .withColumn("row_num", F.row_number().over(windows_spec))
    # Mantendo apenas o primeiro registro (o mais recente) e removendo duplicatas
    .filter(F.col("row_num") == 1)
    # Seleção e renomeação de colunas com transformações de padronização
    .select(
        F.col("seller_id").alias("id_vendedor"),
        F.col("seller_zip_code_prefix").alias("prefixo_cep"),
        # Normalização Geográfica: Transformando cidades e estados em letras maiúsculas 
        F.upper(F.col("seller_city")).alias("cidade"),
        F.upper(F.col("seller_state")).alias("estado"),
        F.col("seller_name").alias("nome_vendedor"),
        # Convertendo a data de registro de texto para o tipo Timestamp para análises temporais
        F.to_timestamp(F.col("seller_registration_date")).alias("data_registro_vendedor")
    )
)
#Usando o overwriteSchema para que o Job não quebre caso a estrutura da Bronze mude
df_silver_sellers.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medalhao.silver.dim_vendedores")


In [0]:
# Importando as funções do PySpark para seleção e transformação de colunas
from pyspark.sql import functions as F

# Carregando a tabela bruta de tradução de categorias da camada Bronze
# Esta tabela serve como um dicionário para converter termos técnicos em nomes amigáveis
df_bronze_category= spark.table("medalhao.bronze.tb_product_category_name_translation")

# Iniciando o mapeamento para a criação da dimensão de tradução na camada Silver
df_silver_category = (
    df_bronze_category
    .select(
        # Renomeando as colunas originais para um padrão em português mais claro 'product_category_name' contém os nomes originais
        F.col("product_category_name").alias("nome_categoria_produto"),
        # 'product_category_name_english' contém a tradução para o inglês
        F.col("product_category_name_english").alias("nome_produto_en")
    )
    # O uso do distinct() é para garantir que cada par de tradução seja único, eliminando qualquer duplicidade que possa ter vindo da carga bruta
    .distinct()
)
#Usando o overwriteSchema para que o Job não quebre caso a estrutura da Bronze mude
df_silver_category.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medalhao.silver.dim_categoria_produtos_traducao")


In [0]:
#Importando a biblioteca Window do pyspark para utilizar o Window Functions para ter apenas o registro mais recente mantido 
from pyspark.sql import Window
from pyspark.sql import functions as F

#Trazendo os dados brutos da camada bronze
df_bronze_geolocation = spark.table("medalhao.bronze.tb_geolocalizacao")

# Realizando a limpeza que será responsável por remover as duplicações do mesmo prefixo de CEP, pois o dataset original possui múltiplas coordenadas para o mesmo prefixo de 5 dígitos e manter essas duplicatas causaria uma multiplicação indevida dos valores de faturamento no dashboard, então escolho o registro mais recente baseado no timestamp de ingestão para garantir a unicidade do CEP.
windows_spec = Window.partitionBy("geolocation_zip_code_prefix").orderBy(F.col("timestamp_ingestion").desc())

#Realizando o mapeamento do dataframe
#O filter de latitude e longitude garante que apenas coordenadas dentro do território brasileiro sejam mantidas 
df_silver_geolocation = (
    df_bronze_geolocation
    .withColumn("row_num", F.row_number().over(windows_spec))
    .filter("row_num == 1")
    .select(
        F.col("geolocation_zip_code_prefix").alias("id_prefixo_cep"),
        F.col("geolocation_lat").alias("latitude"),
        F.col("geolocation_lng").alias("longitude"),
        F.upper(F.col("geolocation_city")).alias("cidade"),
        F.upper(F.col("geolocation_state")).alias("estado")
    )
    .filter(
        (F.col("latitude").between(-34, 6)) & 
        (F.col("longitude").between(-74, -34))
    )
)
#Usando o overwriteSchema para que o Job não quebre caso a estrutura da Bronze mude
df_silver_geolocation.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medalhao.silver.dim_geolocalizacao")

In [0]:
# Importando as bibliotecas de janelas (Window) e as funções do Spark para manipular os dados
from pyspark.sql import Window
from pyspark.sql import functions as F

# Selecionando as colunas de data e valor de compra da bronze, tratando a data para ignorar a hora e o distinct() garante que não teremos cotações duplicadas para o mesmo dia no início do processo
df_cotacao_raw = spark.table("medalhao.bronze.tb_cotacao_dolar").select(
    F.to_date(F.col("dataHoraCotacao")).alias("data_pura"),
    F.col("cotacaoCompra")
).distinct()

# Identificando o período de início e fim da nossa tabela para saber o tamanho do calendário que precisamos criar
range_datas = df_cotacao_raw.select(F.min("data_pura"), F.max("data_pura")).first()
min_data, max_data = range_datas[0], range_datas[1]

# Criando um DataFrame de intervalo que gera todos os dias entre a menor e a maior data, isso serve para não deixar buraco no calendário, como finais de semana e feriados
df_calendario = (
    spark.range(0, (max_data - min_data).days + 1)
    .withColumn("data_referencia", F.date_add(F.lit(min_data), F.col("id").cast("int")))
)

# Unindo o calendário completo com os dados da bronze usando Left Join, assim mantive todos os dias do calendário
df_join_calendario = df_calendario.join(
    df_cotacao_raw, 
    df_calendario.data_referencia == df_cotacao_raw.data_pura, 
    "left"
)

# Criando o filtro de janela que ordena por data para aplicar o preenchimento de dados. O rowsBetween define que olharei o todo do passado até o registro atual
window_ffill = Window.partitionBy(F.lit(1)) \
                     .orderBy("data_referencia") \
                     .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Aplicando a técnica de Forward Fill: o last() com ignorenulls preenche os nulos (fins de semana) com o último valor válido encontrado garante que cada dia do calendário tenha uma taxa de conversão associada
df_silver_cotacao = (
    df_join_calendario
    .withColumn("taxa_ajustada", F.last("cotacaoCompra", ignorenulls=True).over(window_ffill))
    .select(
        F.col("data_referencia").alias("data_cotacao"),
        F.round(F.col("taxa_ajustada"), 4).alias("taxa_conversao")
    )
)
#Usando o overwriteSchema para que o Job não quebre caso a estrutura da Bronze mude
df_silver_cotacao.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medalhao.silver.dim_cotacao_dolar")


In [0]:
display(spark.table("medalhao.silver.dim_cotacao_dolar"))

data_cotacao,taxa_conversao
2016-01-04,4.038
2016-01-05,4.0108
2016-01-06,4.0297
2016-01-07,4.0469
2016-01-08,4.0244
2016-01-09,4.0244
2016-01-10,4.0244
2016-01-11,4.0147
2016-01-12,4.0293
2016-01-13,3.9857


In [0]:
# Importando as funções do Spark para realizar cálculos e manipulações de colunas
from pyspark.sql import functions as F

# Agregando os pagamentos por pedido para garantir que, se houver mais de uma forma de pagamento, o valor seja somado, isso evita que o join duplique as linhas dos pedidos e garante o total financeiro correto por ID
df_pagamentos_agregados = (
    spark.table("medalhao.silver.fat_pagamentos_pedidos")
    .groupBy("id_pedido")
    .agg(F.sum("valor_pagamento").alias("total_brl"))
)

# Carregando a tabela de pedidos e a dimensão de cotação do dólar que foi tratada anteriormente
df_pedidos = spark.table("medalhao.silver.fat_pedidos")
df_dolar = spark.table("medalhao.silver.dim_cotacao_dolar")

# Realizando o cruzamento principal entre Pedidos, Pagamentos e a Cotação do Dólar. O to_date é aplicado na data da compra para alinhar o formato com a tabela de cotação (YYYY-MM-DD)
df_fat_pedido_total = (
    df_pedidos.alias("p")
    .join(df_pagamentos_agregados.alias("pag"), "id_pedido", "inner")
    .join(df_dolar.alias("d"), F.to_date(F.col("p.data_da_compra_do_pedido")) == F.col("d.data_cotacao"), "left")
    .select(
        F.col("p.id_pedido"),
        F.col("p.id_consumidor"),
        F.col("p.status_pedido").alias("status"),
        F.to_date(F.col("p.data_da_compra_do_pedido")).alias("data_pedido"),
        # Arredondando o valor total em reais para 2 casas decimais conforme padrão financeiro
        F.round(F.col("pag.total_brl"), 2).alias("valor_total_pago_brl"),
        # Criando a coluna em dólar através da divisão do valor em real pela taxa de conversão do dia
        F.round(F.col("pag.total_brl") / F.col("d.taxa_conversao"), 2).alias("valor_total_pago_usd")
    )
)
#Usando o overwriteSchema para que o Job não quebre caso a estrutura da Bronze mude
df_fat_pedido_total.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medalhao.silver.fat_pedido_total")

# Executando o OPTIMIZE com ZORDER para organizar fisicamente os dados no disco, isso melhora drasticamente a performance de filtros e buscas por ID de pedido ou Data nas próximas etapas
spark.sql("OPTIMIZE medalhao.silver.fat_pedido_total ZORDER BY (id_pedido, data_pedido)")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,